# Day 5 补充：从字段到矩阵、参数与内存

先读 `docs/DAY05_ARCHITECTURES.md` 的 2.1.9–2.1.11。
本实验只做配置算术，不加载 0.6B 权重，不分配4096×4096的score。
每一步先预测再执行；个人进度仍为 Day 5。

In [ ]:
import os
import sys
from pathlib import Path

root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/config_math.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from learning.config_math import inspect_config
from reasoning_from_scratch.qwen3 import QWEN_CONFIG_06_B
cfg = dict(QWEN_CONFIG_06_B)
report = inspect_config(cfg, tokens=4096)
print(report['dimensions'])

## 1. 2048维输出不等于2048个参数

`nn.Linear(in,out,bias=False)` 的权重为 `[out,in]`，计算 `x @ W.T`。
先算 Q、K、V、out 和 gate/up/down 的 shape，再看输出。
本表不含 Norm、Embedding、LM Head，不能称为模型总参数。

In [ ]:
for name, shape in report['weight_shapes_out_in'].items():
    print(f'{name:12} shape={shape}, parameters={report["parameters"][name]:,}')
print('单层主要投影参数:', f'{report["projection_parameters_per_layer"]:,}')
assert report['weight_shapes_out_in']['W_query'] == [2048, 1024]
assert report['weight_shapes_out_in']['down_proj'] == [1024, 3072]

## 2. 三种不同的内存

假设权重、Cache、score 每元素都是2字节。
score 是单层临时张量，Cache 是所有层的历史状态；不是直接相加就得到进程峰值。
T从4096翻倍到8192，先预测权重、KV、score分别变成几倍。

In [ ]:
larger = inspect_config(cfg, tokens=8192)
for key in ('projection_weight_bytes_all_layers', 'kv_bytes_all_layers', 'prefill_score_bytes_one_layer'):
    before, after = report[key], larger[key]
    print(f'{key}: {before / 2**20:.1f} -> {after / 2**20:.1f} MiB; ratio={after / before:g}')
assert report['kv_bytes_all_layers'] == 448 * 2**20
assert report['prefill_score_bytes_one_layer'] == 512 * 2**20
print('适用范围:', report['scope'])

## 3. 一次改一个字段

先预测 Hkv 从8改4、F从3072改4096分别影响哪些投影。
这里只推导新结构，不能把原checkpoint无条件加载进改变shape的新模型。

In [ ]:
for field, value in (('n_kv_groups', 4), ('hidden_dim', 4096)):
    changed = inspect_config(dict(cfg, **{field: value}), tokens=4096)
    print('\n修改', field, '=', value)
    for name, shape in changed['weight_shapes_out_in'].items():
        if shape != report['weight_shapes_out_in'][name]:
            print(name, report['weight_shapes_out_in'][name], '->', shape)
    print('KV MiB:', changed['kv_bytes_all_layers'] / 2**20)
    print('sharing ratio:', changed['sharing_ratio'])

## 验收与下一步

- Q投影为什么需要 2048×1024 个权重，而非2048个？
- 为什么FFN的主要参数是3EF，而不是2EF？
- T翻倍，权重不变、Cache两倍、prefill score四倍，分别为什么？
- 为什么不能把所有层Cache和单层score都再乘28？

解释清楚后，继续 `learning/day06_online_softmax.ipynb`。